# 02: Co-registro Espacial e Reamostragem (Resampling)

**Objetivo:** Demonstrar o problema da diferença de resolução e dimensões físicas entre CT e PET, como aplicar a reamostragem para um grid de referência comum com `SimpleITK`, e comprovar por que máscaras de segmentação exigem estritamente interpolação por vizinho mais próximo (`sitkNearestNeighbor`).

---

## 1. Importações e Configuração

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

import config
from src import io_utils, registration

print("Módulos importados com sucesso!")

Módulos importados com sucesso!


## 2. O Descasamento de Grades (CT vs PET)

Vamos comparar os metadados espaciais de CT e PET antes de qualquer processamento.

In [2]:
patient_id = "0001"
caminhos = config.get_caminhos(patient_id)

if caminhos["CT"].exists() and caminhos["PET"].exists():
    ct_img = io_utils.load_sitk_image(caminhos["CT"])
    pet_img = io_utils.load_sitk_image(caminhos["PET"])
    
    meta_ct = io_utils.get_spatial_metadata(ct_img)
    meta_pet = io_utils.get_spatial_metadata(pet_img)
    
    print("=" * 60)
    print(f"CT  - Dimensões: {meta_ct['size']} | Spacing: {meta_ct['spacing']}")
    print(f"PET - Dimensões: {meta_pet['size']} | Spacing: {meta_pet['spacing']}")
    print("=" * 60)
    print("Observe a discrepância: o PET possui muito menos voxels e espaçamento maior!")
else:
    print("Arquivos de dados não disponíveis no caminho local.")

CT  - Dimensões: (512, 512, 365) | Spacing: (0.86328125, 0.86328125, 2.5)
PET - Dimensões: (400, 400, 304) | Spacing: (2.0364201068878174, 2.0364201068878174, 3.0)
Observe a discrepância: o PET possui muito menos voxels e espaçamento maior!


## 3. Reamostragem para a Grade do CT (`resample_to_reference`)

Usando o CT como grid anatômico fixo de referência:
- O PET é projetado no mesmo grid com interpolação linear (`sitkLinear`);
- A máscara cardíaca é reamostrada com vizinho mais próximo (`sitkNearestNeighbor`).

In [3]:
if caminhos["CT"].exists() and caminhos["PET"].exists():
    mask_img = io_utils.load_sitk_image(caminhos["Mascara_Cardiaca"]) if caminhos["Mascara_Cardiaca"].exists() else None
    
    # Executa o pipeline de reamostragem
    reg_res = registration.full_pet_ct_registration_pipeline(
        ct_image=ct_img,
        pet_image=pet_img,
        mask_image=mask_img,
        patient_id=patient_id,
        do_rigid_registration=False,  # Padrão: resampling sem distorção
        verbose=True
    )
    
    pet_reg = reg_res["pet_registered"]
    print("\nDimensões do PET reamostrado:", pet_reg.GetSize())
    print("Dimensões do CT de referência:", ct_img.GetSize())
    assert pet_reg.GetSize() == ct_img.GetSize(), "Grids devem ser idênticos!"


  Pipeline de Co-registro PET/CT: 0001

[1/3] Reamostramento PET → grid do CT...

[2/3] Registro Rígido ignorado (apenas resampling de grid).

[3/3] Reamostramento da Máscara...

✅ Pipeline concluído com sucesso!

Dimensões do PET reamostrado: (512, 512, 365)
Dimensões do CT de referência: (512, 512, 365)
